<a href="https://colab.research.google.com/github/sprildo/Gigio_desidentifica_dados_clinicos/blob/main/exemplo_de_uso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Instalar o pacote
!pip install "pt_Gigio_desidentifica @ https://github.com/sprildo/Gigio_desidentifica_dados_clinicos/releases/download/v1.0.0/pt_Gigio_desidentifica-1.0.0-py3-none-any.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 37.4 MB/s eta 0:00:00


In [7]:
# 2. Clonar o repositório para ter acesso à pasta 'scripts'
!git clone https://github.com/sprildo/Gigio_desidentifica_dados_clinicos.git

fatal: destination path 'Gigio_desidentifica_dados_clinicos' already exists and is not an empty directory.


In [8]:
# 3. Adicionar a pasta clonada aos caminhos do sistema do Python
import sys
sys.path.append('/content/Gigio_desidentifica_dados_clinicos')

In [9]:
import spacy
# Importing the handling function from your scripts folder.
from scripts.preprocessamento import tratar_texto

# Carregando o modelo NER
nlp = spacy.load("pt_Gigio_desidentifica")

# 1. Texto original (exemplo com identificadores alterados)
texto_bruto = """# ID: MARIA MIGUEL SOUZA, 52 ANOS, PROCEDENTE DE RIBEIRÃO PRETO, NATURAL DE Pontal
reg: 004567a, contato telefone: (16) 99999-1111

# QUEIXA PRINCIPAL: DISARTRIA E DIFICULDADE PARA DEAMBULAR HÁ 1 DIA.

# HMA: PACIENTE VEM À UNIDADE POR DEMANDA ESPONTÂNEA ACOMPANHADA DA FILHA Juliana REFERINDO QUE ACORDOU ONTEM
COM DISARTRIA E PERDA DE FORÇA EM MEMBRO INFERIOR DIREITO DIFICULTANDO DEAMBULAÇÃO. FILHA REFERE QUE NO DIA
ANTEIROR MÃE ENCONTRAVA-SE BEM, SEM DEFICITS FOCAIS. AINDA ONTEM DE MANHÃ, FILHA LEVOU MÃE AO
PRONTO SOCORRO Hospital Santo Antonio, ONDE FOI AVALIADA POR NEUROLOGISTA, QUE INFORMOU BAIXA PROBABILIDADE
DE SER SECUNDARIO A AVC E ORIENTOU QUE ELA DEVERIA REALIZAR EXAME DE IMAGEM. INFORMA QUE HOJE PERCEBEU PERDA DE FORÇA EM MSD.
REFERE QUE QUINTA FEIRA MEDICOS PRESCREVERAM MORFINA 10MG 4/4H + DIPIRONA PARA CONTROLE ALGICO.
FOI PRESCRITO TAMBEM BISACODIL E LACTULOSE. REFERE CONSTIPAÇÃO DESDE SABADO. DIURESE PRESENTE, SEM PRODUTOS PATOLOGICOS,
SEM DISURIA. REFERE BEIXA ACEITAÇÃO DA DIETA HÁ 2 DIAS. BOA INGESTA DE LIQUIDOS.
REFERE QUE HÁ 30 DIAS REALIZOU FIXACAO PROFILATICA EM TIBIA ESQUERDA POR LESÃO LITICA (META?)

#AP:
1) NÓDULO PULMONAR A DIREITA
- realizada bx por broncoscopia em 17/07, sem intercorrências, aguarda ap
- PET-CT METABOLISMO GLICOLÍTICO EM LINFONODOS DE CADEIA TORÁCICA INTERNA, MEDIASTINAL E HILAR PULMONAR DIREITA,
CERVICAIS DIREITA, REGIÃO INGUINAL DIREITA, LESÕES OSTEOLÍTICAS EM 9º ARCO COSTAL DIREITO, ÍSQUIO E TÍBIA DIREITA.
SE CONFIRMAÇÃO DE SÍTIO PRIMÁRIO PULMONAR PROVÁVEL T1CN3M1C exame número:123485679

2) DPOC GOLD IIIB

3) LESÃO LÍTICA NA TÍBIA DIREITA (META PULMONAR?)
23/07 - FIXAÇÃO PROFILÁTICA DE TIBIA ESQUERDA COM HASTE INTRAMEDULAR

# MUC:
- MORFINA 10MG 4/4H
- DIPIRONA 1G 6/6H SN
- LACTULOSE
- BISACODIL
- ANORO 1 PUFF/DIA

# ALERGIAS: NEGA
Discutido com Prof. Antonio e Dra. Iara, contudas mantidas"""

# 2. text processing (o modelo foi treinado com o texto tratado)
texto_tratado = tratar_texto(texto_bruto)

# 3. NER Model
doc = nlp(texto_tratado)

# 4. Results
print(f"Texto após tratamento: {texto_tratado}\n")
for ent in doc.ents:
    print(f"Entidade: {ent.text} | Categoria: {ent.label_}")


Texto após tratamento: id: maria miguel souza, 52 anos, procedente de ribeirão preto, natural de pontal
reg: 004567a, contato telefone: 16 99999-1111
 queixa principal: disartria e dificuldade para deambular há 1 dia.
 hma: paciente vem à unidade por demanda espontânea acompanhada da filha juliana referindo que acordou ontem
com disartria e perda de força em membro inferior direito dificultando deambulação. filha refere que no dia
anteiror mãe encontrava-se bem, sem deficits focais. ainda ontem de manhã, filha levou mãe ao
pronto socorro hospital santo antonio, onde foi avaliada por neurologista, que informou baixa probabilidade
de ser secundario a avc e orientou que ela deveria realizar exame de imagem. informa que hoje percebeu perda de força em msd.
refere que quinta feira medicos prescreveram morfina 10mg 4/4h + dipirona para controle algico.
foi prescrito tambem bisacodil e lactulose. refere constipação desde sabado. diurese presente, sem produtos patologicos,
sem disuria. refere 

In [11]:
# 5. Masking (performed by the calling application, not by the package): replace each detected entity with its label
texto_mascarado = texto_tratado
for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
    texto_mascarado = texto_mascarado[:ent.start_char] + f"[{ent.label_}]" + texto_mascarado[ent.end_char:]
print(texto_mascarado)

id: [NOME], 52 anos, procedente de [CIDADE], natural de [CIDADE]
reg: [REGISTRO_HOSPITAL], contato telefone: [TELEFONE]
 queixa principal: disartria e dificuldade para deambular há 1 dia.
 hma: paciente vem à unidade por demanda espontânea acompanhada da filha [NOME] referindo que acordou ontem
com disartria e perda de força em membro inferior direito dificultando deambulação. filha refere que no dia
anteiror mãe encontrava-se bem, sem deficits focais. ainda ontem de manhã, filha levou mãe ao
pronto socorro [INSTITUICAO], onde foi avaliada por neurologista, que informou baixa probabilidade
de ser secundario a avc e orientou que ela deveria realizar exame de imagem. informa que hoje percebeu perda de força em msd.
refere que quinta feira medicos prescreveram morfina 10mg 4/4h + dipirona para controle algico.
foi prescrito tambem bisacodil e lactulose. refere constipação desde sabado. diurese presente, sem produtos patologicos,
sem disuria. refere beixa aceitação da dieta há 2 dias. boa 